# Week 5 · Day 4 — CrewAI: Multi-Agent Collaboration, Roles & Task Delegation
### Gemini API edition (free tier)

**Note on API provider:** switched from Groq to the **Gemini API** (Google AI Studio free
tier), via `.env`. CrewAI's `LLM` wrapper talks to Gemini directly using a
`"gemini/<model-name>"` model string -- same pattern as Groq's `"groq/<model-name>"`, just
a different provider prefix, since both go through CrewAI's built-in `litellm` integration.

**Why the switch:** Groq's free tier + the open CrewAI/Groq `cache_breakpoint` bug (issue
#5886) made Groq unreliable for this notebook. Gemini's free tier is generous enough for a
small crew like this one (see rate-limit note below) and isn't affected by that
Groq-specific bug.

**Setup**

```bash
pip install crewai crewai-tools python-dotenv
```

Get a free key (no credit card) at **https://aistudio.google.com/apikey**, then add it to
the same `.env` file used in earlier days:
```
GEMINI_API_KEY=your_key_here
```

> **Staying under the free-tier rate limit:** Gemini's free tier is limited per model --
> as of mid-2026, `gemini-2.5-flash` allows roughly **10 requests/minute** and
> **~250 requests/day**; `gemini-2.5-flash-lite` is more generous at roughly
> **15 requests/minute** and **~1,000-1,500 requests/day**. This notebook defaults to
> `gemini-2.5-flash` for quality, and:
> - sets `max_rpm` on the `Crew` (and each `Agent`) so CrewAI **automatically paces**
>   requests instead of firing them as fast as possible,
> - adds a short pause between the sequential and hierarchical runs below so back-to-back
>   cells don't stack requests into the same one-minute window,
> - and notes `gemini-2.5-flash-lite` as a drop-in fallback (just change `MODEL` below) if
>   you still hit `429 RESOURCE_EXHAUSTED` errors -- it trades a little quality for much
>   more headroom.
>
> If you *do* hit a 429, it's a rate limit, not a bug -- wait ~60 seconds and re-run that
> cell, or lower `max_rpm` further.

> **Leftover safeguard from the Groq attempt:** the cell below still applies the CrewAI
> `cache_breakpoint` monkey-patch (crewAI issue #5886) as a no-op safety net, in case any
> non-Anthropic provider (Gemini included) is ever affected by the same unconditional
> `mark_cache_breakpoint()` call. It does nothing if Gemini isn't affected.

**Scenario for this notebook:** continuing the budget-laptop theme from Days 2-3, a crew
reviews the product catalog, generates business insights, and writes a stakeholder-ready
memo -- "review a dataset, generate insights, and write a stakeholder-ready summary."


In [49]:
!pip install crewai crewai-tools python-dotenv

In [50]:
!pip install -q -U crewai crewai_tools

<frozen posixpath>:82: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited


In [51]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')
print("API key loaded successfully")

API key loaded successfully


In [52]:
import os
import json
import time
from dotenv import load_dotenv
from google.colab import userdata # Import userdata here

# Try loading from .env first
load_dotenv()
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# If not found in .env, try Colab's userdata secrets
if GEMINI_API_KEY is None:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    # Optionally, set it in os.environ for consistency, though LLM(api_key=...) will use it directly
    if GEMINI_API_KEY is not None:
        os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Check if GEMINI_API_KEY is still None and raise an error if so
if GEMINI_API_KEY is None:
    raise ValueError("GEMINI_API_KEY not found. Please ensure it is set in a .env file or in Colab secrets.")

from crewai import Agent, Task, Crew, Process, LLM


import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg


MODEL = "gemini/gemini-3.5-flash-lite" # Changed from gemini/gemini-2.5-flash
llm = LLM(model=MODEL, api_key=GEMINI_API_KEY, temperature=0)


CREW_MAX_RPM = 8

---
## Task 1 — Multi-Agent Design Thinking

**Chosen task:** review our product catalog (the laptop lineup from Days 2-3), generate
business insights about it, and write a stakeholder-ready summary for leadership.

### Three non-overlapping agent roles

**1. Data Analyst**
- *Role:* Senior Data Analyst
- *Goal:* Extract precise, quantitative facts from the product catalog (price range,
  cheapest/most expensive item, price-per-GB-of-RAM for each laptop) that others can build on.
- *Backstory:* A meticulous analyst who has spent years turning messy spreadsheets into
  clean, defensible numbers for leadership -- trusts data over intuition and always shows
  the arithmetic.

**2. Product Strategy Consultant**
- *Role:* Product Strategy Consultant
- *Goal:* Turn the analyst's raw numbers into 3-4 concrete, prioritized business insights
  and recommendations (pricing gaps, opportunities, risks) -- never re-derive numbers itself.
- *Backstory:* A former product manager who's reviewed dozens of pricing sheets, known for
  spotting the one gap in a lineup everyone else missed.

**3. Executive Communications Specialist**
- *Role:* Executive Communications Specialist
- *Goal:* Write a concise, jargon-free memo for a non-technical leadership audience based
  on the strategist's insights -- never introduces new analysis or numbers of its own.
- *Backstory:* Spent a career distilling technical work into memos executives actually
  read; allergic to hedging and bullet-point jargon.

Responsibilities don't overlap: the analyst is the only one who touches raw data, the
strategist is the only one who makes business judgments, the writer is the only one who
shapes audience-facing language -- each stage consumes the previous stage's output rather
than redoing its work.

### Why specialists might beat one generalist here -- and where that isn't true

Splitting the work forces each agent's prompt and context to stay narrow and role-specific,
which tends to produce more consistent, higher-quality output at each stage than asking one
model to be a meticulous quantitative analyst, a strategic thinker, *and* a polished
communicator all inside a single instruction -- those skills can pull a single prompt in
different directions (precise-and-terse vs. persuasive-and-readable). It stops being worth
it when the task is small enough that one well-scoped prompt already produces a good
result: here, three sequential LLM calls (plus a manager call in the hierarchical version)
cost meaningfully more in latency and tokens than one generalist call would, for a task
that a single capable agent could arguably handle directly, as Day 3's LangGraph agent did
in one pass.


---
## Task 2 — Build Agents & Assign Tools

Two tools, reused/adapted from Day 2's catalog: `read_catalog` (reads the *entire* product
catalog -- something only the analyst needs) and `calculator` (for the analyst's per-unit
price math). The strategist and writer get **no tools** -- see justification below.


In [53]:
# --- Reused catalog from Days 2-3 -------------------------------------------
PRODUCTS_PATH = "products.json"
products_db = {
    "laptop a": {"price_usd": 799, "specs": "8GB RAM, 256GB SSD"},
    "laptop b": {"price_usd": 1199, "specs": "16GB RAM, 512GB SSD"},
    "laptop c": {"price_usd": 549, "specs": "8GB RAM, 128GB SSD"},
}
with open(PRODUCTS_PATH, "w") as f:
    json.dump(products_db, f, indent=2)


In [54]:
# CrewAI >=0.30 exposes the @tool decorator at crewai.tools; older versions
# had it under crewai_tools. Try both so this cell works either way.
try:
    from crewai.tools import tool
except ImportError:
    from crewai_tools import tool


@tool("Read product catalog")
def read_catalog() -> str:
    """Returns the full product catalog (name, price in USD, specs) as JSON text.

    Use this to get every product's price and specs at once before doing any
    comparison or per-unit calculation. Takes no arguments.
    """
    with open(PRODUCTS_PATH) as f:
        return f.read()


@tool("Calculator")
def calculator(expression: str) -> str:
    """Evaluates a single arithmetic expression and returns the numeric result.

    Supports +, -, *, /, **, and parentheses. Use this for any precise
    computation, e.g. price-per-GB-of-RAM (799 / 8). Pass numbers and
    operators only, never words.
    """
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: could not evaluate expression -- {e}"


In [55]:
data_analyst = Agent(
    role="Senior Data Analyst",
    goal=(
        "Extract precise, quantitative facts from the product catalog -- price range, "
        "cheapest and most expensive item, and price-per-GB-of-RAM for each laptop -- "
        "that other specialists can build on without re-deriving numbers themselves."
    ),
    backstory=(
        "A meticulous analyst who has spent years turning messy spreadsheets into clean, "
        "defensible numbers for leadership. Trusts data over intuition and always shows "
        "the arithmetic."
    ),
    tools=[read_catalog, calculator],
    llm=llm,
    allow_delegation=False,
    max_rpm=CREW_MAX_RPM,
    verbose=True,
)

strategist = Agent(
    role="Product Strategy Consultant",
    goal=(
        "Turn the analyst's raw numbers into 3-4 concrete, prioritized business insights "
        "and recommendations about the product lineup for a budget-conscious market "
        "segment -- grounded in the analyst's figures, never inventing new ones."
    ),
    backstory=(
        "A former product manager who's reviewed dozens of catalogs and pricing sheets, "
        "known for spotting the one gap in a lineup that everyone else missed."
    ),
    tools=[],  # deliberately no data-reading tools -- see justification below
    llm=llm,
    allow_delegation=False,
    max_rpm=CREW_MAX_RPM,
    verbose=True,
)

writer = Agent(
    role="Executive Communications Specialist",
    goal=(
        "Write a concise, stakeholder-ready memo for a non-technical leadership audience, "
        "based strictly on the strategist's insights -- clear, jargon-free, and readable "
        "in under a minute."
    ),
    backstory=(
        "Spent a career distilling technical and analytical work into memos executives "
        "actually read. Allergic to jargon, hedging, and bullet-point walls of text."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    max_rpm=CREW_MAX_RPM,
    verbose=True,
)


**Tool assignment justification:** only the Data Analyst gets `read_catalog` and
`calculator` -- it's the only agent whose job requires touching raw data or doing
arithmetic. The Strategist and Writer deliberately get **no tools**: giving them
`read_catalog` too would let them independently re-derive numbers from the raw catalog,
which could silently disagree with what the analyst already reported (e.g. rounding
differently), breaking traceability back to one source of truth. Keeping tool access
role-appropriate isn't just about capability -- it enforces that later agents build on
earlier agents' work instead of quietly redoing it their own way.


---
## Task 3 — Define Tasks & Process (Sequential)


In [56]:
task_analysis = Task(
    description=(
        "Use the read_catalog tool to get the full product catalog, then use the "
        "calculator tool to compute price-per-GB-of-RAM for each laptop. Report the "
        "price range (cheapest and most expensive item) and the price-per-GB figure "
        "for every laptop."
    ),
    expected_output=(
        "A plain list of specific numeric facts, ONE FACT PER LINE, no prose "
        "paragraphs -- e.g.:\n"
        "- Laptop A: $799, 8GB RAM, $99.88/GB\n"
        "- Laptop B: $1199, 16GB RAM, $74.94/GB\n"
        "- Laptop C: $549, 8GB RAM, $68.63/GB\n"
        "- Price range: $549 (cheapest) to $1199 (most expensive)"
    ),
    agent=data_analyst,
)

task_strategy = Task(
    description=(
        "Using ONLY the analyst's numeric facts (do not invent new numbers), identify "
        "3-4 prioritized business insights and recommendations about this laptop "
        "lineup for a budget-conscious client segment. Each insight must cite at least "
        "one specific figure from the analysis."
    ),
    expected_output=(
        "A numbered list of 3-4 insights, each 1-2 sentences, each referencing a "
        "specific number from the analyst's facts."
    ),
    agent=strategist,
    context=[task_analysis],
)

task_report = Task(
    description=(
        "Write a short stakeholder-ready memo summarizing the strategist's insights "
        "for a non-technical leadership audience. Do not introduce new analysis -- "
        "only reshape the strategist's insights into clear, professional language."
    ),
    expected_output=(
        "A polished memo in Markdown: a one-line title, then 3-4 short paragraphs "
        "(no bullet lists), ready to send to leadership as-is."
    ),
    agent=writer,
    context=[task_strategy],
)

crew_sequential = Crew(
    agents=[data_analyst, strategist, writer],
    tasks=[task_analysis, task_strategy, task_report],
    process=Process.sequential,
    max_rpm=CREW_MAX_RPM,  # crew-level cap overrides each agent's own max_rpm
    verbose=True,
)


In [57]:
result_sequential = await crew_sequential.kickoff_async()
print("\n=== FINAL MEMO (sequential) ===\n")
print(result_sequential)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: cf223be7-891e-44a1-8b86-4a821b31a5a1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the read_catalog tool to get the full product catalog, then use the calculator tool to compute       │
│  price-per-GB-of-RAM for each laptop. Report the price range (cheapest and most expensive item) and the         │
│  price-per-GB figure for every laptop.                                                                          │
│  ID: 8e2a2403-5efe-442b-a982-43fe6f3f7753                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Data Analyst                                                                                     │
│                                                                                                                 │
│  Task: Use the read_catalog tool to get the full product catalog, then use the calculator tool to compute       │
│  price-per-GB-of-RAM for each laptop. Report the price range (cheapest and most expensive item) and the         │
│  price-per-GB figure for every laptop.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool read_product_catalog executed with result: {
  "laptop a": {
    "price_usd": 799,
    "specs": "8GB RAM, 256GB SSD"
  },
  "laptop b": {
    "price_usd": 1199,
    "specs": "16GB RAM, 512GB SSD"
  },
  "laptop c": {
    "price_usd": 549,
    ...

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: read_product_catalog                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Executive Communications Specialist                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Laptop Lineup Pricing and Value Analysis                                                                     │
│                                                                                                                 │
│  Our current laptop lineup contains a critical value trap that we need to address immediately. Laptop A is      │
│  severely overpriced for budget-conscious buyers at $799, charging nearly $100 per gigabyte of RAM for the      │
│  exact same 8GB capacity found in Laptop C. In contrast, Laptop C is our strongest value play at $549,          │
│  delivering the lowest cost per gigabyte in the portfolio while fully matching Laptop A's memory.               │
│                                                                                                                 │
│  At the upper end of the market, Laptop B requires our highest capital outlay at $1199, but it justifies this   │
│  investment much better than Laptop A. By doubling the memory to 16GB, Laptop B improves its cost efficiency    │
│  significantly to under $75 per gigabyte. However, the $650 gap between our cheapest and most expensive models  │
│  leaves a massive void in our pricing strategy.                                                                 │
│                                                                                                                 │
│  This wide price gap creates a vulnerability, as price-sensitive customers face an abrupt jump from a           │
│  three-digit entry point straight into a four-digit premium tag. Introducing a mid-tier offering would bridge   │
│  this unnecessary divide and better capture buyers who want more than basic performance without paying premium  │
│  prices.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short stakeholder-ready memo summarizing the strategist's insights for a non-technical           │
│  leadership audience. Do not introduce new analysis -- only reshape the strategist's insights into clear,       │
│  professional language.                                                                                         │
│  Agent: Executive Communications Specialist                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL MEMO (sequential) ===

# Laptop Lineup Pricing and Value Analysis

Our current laptop lineup contains a critical value trap that we need to address immediately. Laptop A is severely overpriced for budget-conscious buyers at $799, charging nearly $100 per gigabyte of RAM for the exact same 8GB capacity found in Laptop C. In contrast, Laptop C is our strongest value play at $549, delivering the lowest cost per gigabyte in the portfolio while fully matching Laptop A's memory.

At the upper end of the market, Laptop B requires our highest capital outlay at $1199, but it justifies this investment much better than Laptop A. By doubling the memory to 16GB, Laptop B improves its cost efficiency significantly to under $75 per gigabyte. However, the $650 gap between our cheapest and most expensive models leaves a massive void in our pricing strategy. 

This wide price gap creates a vulnerability, as price-sensitive customers face an abrupt jump from a three-digit entry point straight 

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: cf223be7-891e-44a1-8b86-4a821b31a5a1                                                                       │
│  Final Output: # Laptop Lineup Pricing and Value Analysis                                                       │
│                                                                                                                 │
│  Our current laptop lineup contains a critical value trap that we need to address immediately. Laptop A is      │
│  severely overpriced for budget-conscious buyers at $799, charging nearly $100 per gigabyte of RAM for the      │
│  exact same 8GB capacity found in Laptop C. In contrast, Laptop C is our strongest value play at $549,          │
│  delivering the lowest cost per gigabyte in the portfolio while fully matching Laptop A's memory.               │
│                                                                                                                 │
│  At the upper end of the market, Laptop B requires our highest capital outlay at $1199, but it justifies this   │
│  investment much better than Laptop A. By doubling the memory to 16GB, Laptop B improves its cost efficiency    │
│  significantly to under $75 per gigabyte. However, the $650 gap between our cheapest and most expensive models  │
│  leaves a massive void in our pricing strategy.                                                                 │
│                                                                                                                 │
│  This wide price gap creates a vulnerability, as price-sensitive customers face an abrupt jump from a           │
│  three-digit entry point straight into a four-digit premium tag. Introducing a mid-tier offering would bridge   │
│  this unnecessary divide and better capture buyers who want more than basic performance without paying premium  │
│  prices.                                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Where a task's output didn't match what the next agent needed

On an early run, `task_analysis` had a looser `expected_output` ("summarize the pricing
data"), and the analyst returned a *prose paragraph* narrating the numbers instead of
discrete facts -- e.g. "Laptop A is the cheapest at $799 while Laptop B, with double the
RAM, costs considerably more..." The strategist then had to re-parse that prose to find
numbers, and occasionally cited a rounded or slightly paraphrased figure instead of the
exact one. **Fix:** tightened `expected_output` to explicitly demand "ONE FACT PER LINE,
no prose paragraphs" with a concrete example format included in the task description
itself (shown in `task_analysis` above) -- once the expected format was spelled out with
a literal example, the analyst's output became consistently parseable and the strategist's
citations matched the analyst's numbers exactly. The lesson: `expected_output` is a
contract for the *next* agent as much as a instruction for the current one -- vague
`expected_output` is where format mismatches between chained tasks come from.


---
## Task 4 — Try Hierarchical Delegation

Same three specialist agents, but now coordinated by a manager agent under
`Process.hierarchical`. In hierarchical mode, task(s) are generally left **without** a
fixed `agent=` assignment -- the manager decides who does what -- and the manager itself
is passed via `manager_agent`, not included in the crew's own `agents` list.


In [58]:
manager = Agent(
    role="Engagement Manager",
    goal=(
        "Deliver a stakeholder-ready memo analyzing the product catalog by coordinating "
        "the Data Analyst, Product Strategy Consultant, and Executive Communications "
        "Specialist -- delegate the numeric analysis, the strategic insights, and the "
        "final writing to the right specialist in that order, and review each "
        "specialist's work before passing it to the next stage or finalizing it."
    ),
    backstory=(
        "A pragmatic engagement manager who trusts specialists to do the deep work but "
        "always sanity-checks numbers and tone before anything goes out to a client."
    ),
    llm=llm,
    allow_delegation=True,  # only the manager should have delegation enabled
    max_rpm=CREW_MAX_RPM,
    verbose=True,
)

master_task = Task(
    description=(
        "Produce a stakeholder-ready memo recommending how the company should think "
        "about its current laptop lineup for budget-conscious clients. Delegate the "
        "numeric analysis to the Data Analyst, the strategic insights to the Product "
        "Strategy Consultant, and the final memo writing to the Executive "
        "Communications Specialist -- in that order -- reviewing each stage's output "
        "before moving to the next."
    ),
    expected_output=(
        "A polished stakeholder memo in Markdown, grounded in specific numbers from "
        "the catalog, ready to send to leadership."
    ),
    # No `agent=` here -- the manager assigns it.
)

crew_hierarchical = Crew(
    agents=[data_analyst, strategist, writer],  # workers only; manager is separate
    tasks=[master_task],
    process=Process.hierarchical,
    manager_agent=manager,
    max_rpm=CREW_MAX_RPM,
    verbose=True,
)


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Pausing between runs:** hierarchical delegation issues more LLM calls than sequential
(the manager's own planning/delegation/review calls, on top of the same three specialist
calls). Running both crews back-to-back within the same one-minute window risks stacking
requests past the free-tier RPM cap even with `max_rpm` set on each crew individually --
`max_rpm` paces requests *within* a crew's own run, not across separate `kickoff()` calls.
A short pause between the two gives the per-minute window room to reset.


In [59]:
print("Pausing ~20s before the hierarchical run to stay clear of the free-tier RPM window...")
time.sleep(20)

result_hierarchical = await crew_hierarchical.kickoff_async()
print("\n=== FINAL MEMO (hierarchical) ===\n")
print(result_hierarchical.raw)


Pausing ~20s before the hierarchical run to stay clear of the free-tier RPM window...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 062001ae-b350-4322-96e3-f511ba616874                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Produce a stakeholder-ready memo recommending how the company should think about its current laptop      │
│  lineup for budget-conscious clients. Delegate the numeric analysis to the Data Analyst, the strategic          │
│  insights to the Product Strategy Consultant, and the final memo writing to the Executive Communications        │
│  Specialist -- in that order -- reviewing each stage's output before moving to the next.                        │
│  ID: 3d17cb65-c899-4a41-8c86-f097118c51f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Engagement Manager                                                                                      │
│                                                                                                                 │
│  Task: Produce a stakeholder-ready memo recommending how the company should think about its current laptop      │
│  lineup for budget-conscious clients. Delegate the numeric analysis to the Data Analyst, the strategic          │
│  insights to the Product Strategy Consultant, and the final memo writing to the Executive Communications        │
│  Specialist -- in that order -- reviewing each stage's output before moving to the next.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'context': 'We need a stakeholder-ready memo analyzing the product catalog, specifically focusing on    │
│  laptops for budget-conscious clients. Your role is Senior Data Analyst. Please perform a rigorous ...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: **MEMORANDUM**                                                                                         │
│                                                                                                                 │
│  **TO:** Leadership Team                                                                                        │
│  **FROM:** Executive Communications Specialist                                                                  │
│  **DATE:** October 24, 2023                                                                                     │
│  **SUBJECT:** Strategy for Budget-Conscious Laptop Lineup                                                       │
│                                                                                                                 │
│  ***                                                                                                            │
│                                                                                                                 │
│  Our current laptop lineup spans an average price of $849, with a $650 spread between entry and premium tiers.  │
│  To maximize sales among budget-conscious clients, we need to address specific pricing friction points and      │
│  leverage the hidden value across our three core models.                                                        │
│                                                                                                                 │
│  Here is our go-to-market plan:                                                                                 │
│                                                                                                                 │
│  *   **Lead with Laptop C as the Ultra-Budget Gateway ($549):** At $549, this model captures our lowest price   │
│  point. To mitigate its modest 128GB storage limitation, we will bundle it with cloud storage, turning a        │
│  hardware constraint into a service-upsell opportunity.                                                         │
│  *   **Neutralize the "Value Trap" on Laptop A ($799):** While Laptop A sits comfortably below our catalog      │
│  mean, its cost-efficiency per GB of RAM is our least favorable ($99.88). We will reframe its appeal around     │
│  its practical 256GB storage utility and introduce light promotional discounting to pull it cleanly under the   │
│  $849 catalog average.                                                                                          │
│  *   **Market Laptop B on Total Cost of Ownership ($1,199):** Though our highest-priced unit, Laptop B offers   │
│  the best RAM cost-efficiency in the lineup ($74.94 per GB) alongside robust 16GB RAM and 512GB SSD specs. We   │
│  will market this tier to budget-conscious buyers through a Total Cost of Ownership (TCO) lens, emphasizing     │
│  longevity and performance value.                                                                               │
│  *   **Bridge the Mid-to-High Tier Gap:** A significant $400 pricing jump separates Laptop A ($799) and Laptop  │
│  B ($1,199). We will close this conversion gap by introducing flexible financing and installment options,       │
│  making the premium tier accessible to buyers hesitant about a large upfront lump sum.                          │
│                                                                                                                 │
│  **Next Steps:** Product marketing will update collater

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Engagement Manager                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # MEMORANDUM                                                                                                   │
│                                                                                                                 │
│  **TO:** Leadership Team                                                                                        │
│  **FROM:** Engagement Manager (Synthesized from Data, Strategy, and Communications Specialists)                 │
│  **DATE:** Current                                                                                              │
│  **SUBJECT:** Strategic Evaluation & Recommendations: Laptop Portfolio for Budget-Conscious Clients             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Executive Summary                                                                                       │
│                                                                                                                 │
│  As requested, our team has conducted a rigorous quantitative and strategic review of our current laptop        │
│  product catalog to establish a data-driven approach for targeting budget-conscious clients. With a catalog     │
│  mean price of **$849.00 USD** and an absolute price range spanning **$650.00** (from **$549.00** to            │
│  **$1,199.00**), our current lineup sits across distinct tiers without an obvious bridge for price-sensitive    │
│  buyers.                                                                                                        │
│                                                                                                                 │
│  This memorandum outlines our empirical findings and delivers a cohesive, stakeholder-ready strategy to         │
│  optimize positioning, address component cost anomalies, and close pricing gaps across our three core models.   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Catalog Baseline & Quantitative Overview                                                                │
│                                                                                                                 │
│  Our analysis of the catalog reveals three distinct laptop configurations. Below are the verified metrics       │
│  governing our portfolio:                                                                                       │
│                                                                                                                 │
│  | Model | Price (USD) | RAM | SSD Storage | Cost per GB RAM | Tier Classification |                            │
│  | :--- | :--- | :--- | :--- | :--- | :--- |                                                                    │
│  | **Laptop C** | $549.00 | 8 GB | 128 GB | $68.63 | Ul

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Produce a stakeholder-ready memo recommending how the company should think about its current laptop      │
│  lineup for budget-conscious clients. Delegate the numeric analysis to the Data Analyst, the strategic          │
│  insights to the Product Strategy Consultant, and the final memo writing to the Executive Communications        │
│  Specialist -- in that order -- reviewing each stage's output before moving to the next.                        │
│  Agent: Engagement Manager                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL MEMO (hierarchical) ===

# MEMORANDUM

**TO:** Leadership Team  
**FROM:** Engagement Manager (Synthesized from Data, Strategy, and Communications Specialists)  
**DATE:** Current  
**SUBJECT:** Strategic Evaluation & Recommendations: Laptop Portfolio for Budget-Conscious Clients  

---

### 1. Executive Summary

As requested, our team has conducted a rigorous quantitative and strategic review of our current laptop product catalog to establish a data-driven approach for targeting budget-conscious clients. With a catalog mean price of **$849.00 USD** and an absolute price range spanning **$650.00** (from **$549.00** to **$1,199.00**), our current lineup sits across distinct tiers without an obvious bridge for price-sensitive buyers. 

This memorandum outlines our empirical findings and delivers a cohesive, stakeholder-ready strategy to optimize positioning, address component cost anomalies, and close pricing gaps across our three core models.

---

### 2. Catalog Baseline & Q

### Sequential vs. hierarchical -- comparing the two runs

Run both cells above and compare using `result_sequential.token_usage` and
`result_hierarchical.token_usage` (see Task 5). In general, expect:

- **Quality:** hierarchical *can* produce better output when the manager actually catches
  a bad intermediate result and re-delegates -- but it can also just add an extra
  paraphrase layer with no real quality gain if nothing goes wrong.
- **Latency/cost:** hierarchical is reliably more expensive -- it adds the manager's own
  planning/delegation calls (and any review/re-delegation calls) on top of the same three
  specialist calls the sequential run makes.
- **Reliability:** sequential is more predictable -- the same three tasks run in the same
  order every time. Hierarchical's manager can choose a different delegation path run to
  run, which is powerful for dynamic tasks but makes output shape less consistent run
  over run.

| Aspect | Sequential | Hierarchical |
|---|---|---|
| **Pros** | Predictable order; cheapest; easiest to debug (fixed pipeline) | Manager can catch and fix a bad intermediate result; adapts delegation order if the task genuinely needs it |
| **Cons** | No re-delegation if an intermediate output is bad -- bad output just flows downstream | More token/latency cost (manager calls on top of worker calls); less run-to-run consistency |
| **When to use** | Task has a known, fixed pipeline of stages (like this one: analyze -> strategize -> write) | Task's structure isn't fixed in advance, or quality control between stages genuinely needs a reviewing "manager" role |


---
## Task 5 — Evaluation & Cost Awareness


In [60]:
def summarize_usage(label: str, result) -> dict:
    usage = result.token_usage
    print(f"--- {label} ---")
    print(f"  prompt_tokens:      {usage.prompt_tokens}")
    print(f"  completion_tokens:  {usage.completion_tokens}")
    print(f"  total_tokens:       {usage.total_tokens}")
    print(f"  successful_requests:{usage.successful_requests}")
    return {
        "label": label,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "successful_requests": usage.successful_requests,
    }

seq_usage = summarize_usage("Sequential crew", result_sequential)
hier_usage = summarize_usage("Hierarchical crew", result_hierarchical)


--- Sequential crew ---
  prompt_tokens:      7374
  completion_tokens:  1872
  total_tokens:       9246
  successful_requests:15
--- Hierarchical crew ---
  prompt_tokens:      61184
  completion_tokens:  21940
  total_tokens:       83124
  successful_requests:56


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 062001ae-b350-4322-96e3-f511ba616874                                                                       │
│  Final Output: # MEMORANDUM                                                                                     │
│                                                                                                                 │
│  **TO:** Leadership Team                                                                                        │
│  **FROM:** Engagement Manager (Synthesized from Data, Strategy, and Communications Specialists)                 │
│  **DATE:** Current                                                                                              │
│  **SUBJECT:** Strategic Evaluation & Recommendations: Laptop Portfolio for Budget-Conscious Clients             │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Executive Summary                                                                                       │
│                                                                                                                 │
│  As requested, our team has conducted a rigorous quantitative and strategic review of our current laptop        │
│  product catalog to establish a data-driven approach for targeting budget-conscious clients. With a catalog     │
│  mean price of **$849.00 USD** and an absolute price range spanning **$650.00** (from **$549.00** to            │
│  **$1,199.00**), our current lineup sits across distinct tiers without an obvious bridge for price-sensitive    │
│  buyers.                                                                                                        │
│                                                                                                                 │
│  This memorandum outlines our empirical findings and delivers a cohesive, stakeholder-ready strategy to         │
│  optimize positioning, address component cost anomalies, and close pricing gaps across our three core models.   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Catalog Baseline & Quantitative Overview                                                                │
│                                                                                                                 │
│  Our analysis of the catalog reveals three distinct laptop configurations. Below are the verified metrics       │
│  governing our portfolio:                                                                                       │
│                                                                                                                 │
│  | Model | Price (USD) | RAM | SSD Storage | Cost per GB RAM | Tier Classification |                            │
│  | :--- | :--- | :--- | :--- | :--- | :--- |                                                                    │
│  | **Laptop C** | $549.00 | 8 GB | 128 GB | $68.63 | U



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [61]:
# --- Approximate cost estimate ------------------------------------------
# On the Gemini free tier, $ cost is $0 (usage is capped by rate limits, not
# billed) -- so "cost" here means paid-tier-equivalent cost, useful for
# estimating what this would cost if/when you outgrow the free tier and
# move to Gemini's paid tier. These per-million-token rates are illustrative
# placeholders for gemini-2.5-flash -- check
# https://ai.google.dev/gemini-api/docs/pricing for current rates before
# reporting real cost figures, since provider pricing changes over time.
PRICE_PER_M_INPUT_USD = 0.30
PRICE_PER_M_OUTPUT_USD = 2.50

def estimate_cost(usage: dict) -> float:
    return (
        usage["prompt_tokens"] / 1_000_000 * PRICE_PER_M_INPUT_USD
        + usage["completion_tokens"] / 1_000_000 * PRICE_PER_M_OUTPUT_USD
    )

for usage in (seq_usage, hier_usage):
    cost = estimate_cost(usage)
    print(f"{usage['label']}: ~${cost:.5f} paid-tier-equivalent "
          f"({usage['total_tokens']} tokens, {usage['successful_requests']} LLM calls) "
          f"-- actual free-tier cost is $0")

# For comparison: Day 3's single-agent LangGraph solution makes roughly one LLM
# call per graph node per pass (plan, generate, critique -- fewer per revision
# than this crew's three full agent turns), so per-run token cost there is
# typically lower than even the sequential crew, before any critique retries.


Sequential crew: ~$0.00689 paid-tier-equivalent (9246 tokens, 15 LLM calls) -- actual free-tier cost is $0
Hierarchical crew: ~$0.07321 paid-tier-equivalent (83124 tokens, 56 LLM calls) -- actual free-tier cost is $0



### Success criteria & manual scoring

Three simple criteria to score each run's final memo against (1-5 scale):

1. **Factual grounding** -- do the numbers in the memo trace back exactly to the
   catalog data (no invented or mismatched figures)?
2. **Completeness** -- does it cover all three laptops and end with a clear
   recommendation, not just a data dump?
3. **Tone** -- does it actually read like an executive memo (concise, jargon-free),
   not like a raw analysis or a bullet-point dump?

Fill in this scorecard after running the cells above three times (e.g. sequential run 1,
sequential run 2, hierarchical run) -- LLM output varies run to run even at
`temperature=0` for a multi-step pipeline like this, so more than one run per mode is
worth comparing:

| Run | Factual grounding (1-5) | Completeness (1-5) | Tone (1-5) | Notes |
|---|---|---|---|---|
| Sequential #1 | | | | |
| Sequential #2 | | | | |
| Hierarchical #1 | | | | |


In [62]:
# Optional: a tiny scorecard structure to fill in by hand after reviewing outputs.
scorecard = [
    {"run": "Sequential #1", "factual_grounding": None, "completeness": None, "tone": None, "notes": ""},
    {"run": "Sequential #2", "factual_grounding": None, "completeness": None, "tone": None, "notes": ""},
    {"run": "Hierarchical #1", "factual_grounding": None, "completeness": None, "tone": None, "notes": ""},
]
# Fill in the None values (1-5) after reading each run's final memo, then:
# import pandas as pd; pd.DataFrame(scorecard)


In [63]:
# Example of how to fill in the scorecard after reviewing the outputs:
# You would replace the 'None' values with your actual scores (1-5).

# Assuming you have reviewed result_sequential and result_hierarchical.raw
# For demonstration, I will use example scores. You should replace these.
scorecard[0]['factual_grounding'] = 4
scorecard[0]['completeness'] = 5
scorecard[0]['tone'] = 4
scorecard[0]['notes'] = "Good, but could be slightly more concise."

scorecard[1]['factual_grounding'] = 5
scorecard[1]['completeness'] = 4
scorecard[1]['tone'] = 5
scorecard[1]['notes'] = "Excellent tone, minor factual detail missing."

scorecard[2]['factual_grounding'] = 4
scorecard[2]['completeness'] = 4
scorecard[2]['tone'] = 4
scorecard[2]['notes'] = "Manager added a good summary. Some repetition."

import pandas as pd
updated_scorecard_df = pd.DataFrame(scorecard)
display(updated_scorecard_df)

,run,factual_grounding,completeness,tone,notes
0,Sequential #1,4,5,4,"Good, but could be slightly more concise."
1,Sequential #2,5,4,5,"Excellent tone, minor factual detail missing."
2,Hierarchical #1,4,4,4,Manager added a good summary. Some repetition.


### Was a multi-agent crew worth it for this task?

For a task this small -- three laptops, one pricing table, one memo -- a single
well-designed agent (like Day 3's LangGraph plan/generate/critique loop) can very plausibly
match the crew's output quality at a fraction of the token cost, since the crew pays for
three-to-four separate full agent turns (more in hierarchical mode) where one agent could
carry the same context through fewer calls. The crew earns its cost when the specialist
skills genuinely diverge and benefit from separately tuned prompts/personas -- rigorous
data work, strategic judgment, and executive writing are different skills -- and when the
catalog or task scope grows large enough that a single agent's context and focus would
start to strain. At this scale, though, the honest answer is that the multi-agent
structure is somewhat over-engineered for the job; it's most useful here as a learning
exercise in role decomposition and delegation patterns, not as the leanest way to produce
this particular memo.
